In [1]:
import numpy as np
import pandas as pd
import scanpy as sc

import torch
from torch import nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import callbacks 

from src import PATH

In [2]:
from torch.utils.data import DataLoader, Dataset, TensorDataset

In [3]:
from torch.utils.data import DataLoader, Dataset, TensorDataset

In [4]:
adata_diff = sc.read_h5ad(
    "/home/wergillius/Project/diffuse_differentiate/data/TFAtlas/GSE217460_210322_TFAtlas_differentiated.h5ad"
)
adata_diff

AnnData object with n_obs × n_vars = 28825 × 4806
    obs: 'TF', 'batch', 'louvain', 'n_counts', 'n_genes', 'percent_mito', 'dpt_pseudotime', 'discrete_time', 'split', 'cell_type', 'Cluster Enriched TFs_colors', 'highlight_celltype', 'Depth_from_root', 'Cluster Enriched TFs'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'Cluster Enriched TFs_colors', 'KNN', 'batch_colors', 'diffmap_evals', 'highlight_celltype_colors', 'hvg', 'iroot', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'umap', 'unique_token_dict'
    obsm: 'X_diffmap', 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    layers: 'a2_PathSampled_X', 'a3_PathSampled_X'
    obsp: 'KNN_connectivities', 'KNN_distances', 'connectivities', 'distances'

# Tensor ds

In [5]:
TF_onehot = pd.get_dummies(adata_diff.obs.TF).values
gene_X = adata_diff.X
X =  np.concatenate([gene_X, TF_onehot], axis=1)

Y = adata_diff.layers['a3_PathSampled_X'].copy()

train_idx = adata_diff.obs.split != 'test'
test_idx = adata_diff.obs.split == 'test'

x_train = torch.from_numpy(X[train_idx]).float()
x_test = torch.from_numpy(X[test_idx]).float()

Y_train = torch.from_numpy(Y[train_idx]).float()
Y_test = torch.from_numpy(Y[test_idx]).float()

In [6]:
train_ds = TensorDataset(x_train, Y_train)
train_dl = DataLoader(train_ds, batch_size=32, num_workers=8, shuffle=True)

test_ds = TensorDataset(x_test, Y_test)
test_dl = DataLoader(test_ds, batch_size=32,  num_workers=8, shuffle=False)

In [7]:
x_train.shape

torch.Size([25942, 7341])

# define module

In [8]:
from importlib import reload
from src import _helper_net

In [9]:
reload(_helper_net)

<module 'src._helper_net' from '/home/wergillius/Project/diffuse_differentiate/src/_helper_net.py'>

In [10]:
from scipy.stats import pearsonr

def evaluate_r(test_Y, pred_Y):

    r_dict = []
    for i, gene in enumerate(adata_diff.var.index):
        r_dict.append(
            {"gene":gene, 'r2': pearsonr(test_Y[:,i], pred_Y[:,i])[0]**2}
        )

    return pd.json_normalize(r_dict)

In [36]:
class Linear_model(pl.LightningModule):
    def __init__(self, input_dim, output_dim, lr, weight_decay):
        super().__init__()

        self.save_hyperparameters()

        self.model = nn.Linear(input_dim, output_dim)
        self.lr = lr
        self.weight_decay = weight_decay

        self.loss_fn = nn.MSELoss()

        self.train_r2_score = torchmetrics.R2Score(num_outputs = output_dim)
        self.val_r2_score = torchmetrics.R2Score(num_outputs = output_dim)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        return optimizer
    
    def forward(self,X):
        return self.model(X)
    
    def training_step(self, train_batch, batch_idx):
        X,Y = train_batch
        Ypred= self.forward(X)
        loss = self.loss_fn(Y, Ypred)
        r2 = self.train_r2_score(Y, Ypred)
        self.log_dict({'train_loss':loss, 'train_r2':r2})
        return loss

    def validation_step(self, val_batch, batch_idx):
        X,Y = val_batch
        Ypred= self.forward(X)
        loss = self.loss_fn(Y, Ypred)
        r2 = self.val_r2_score(Y,Ypred)
        self.log_dict({'val_loss':loss, 'val_r2':r2})
        return loss

    def test_step(self, test_batch, batch_idx):
        X,Y = test_batch
        Ypred= self.forward(X)
        loss = self.loss_fn(Y, Ypred)
        self.log_dict({'test_loss':loss})
        return loss
    

class CAE_model(Linear_model):
    def __init__(self, input_dim, output_dim, hidden, lr, weight_decay):
        super().__init__(input_dim, output_dim, lr, weight_decay)

        if type(hidden) == int:
            hidden = [hidden]
        bottlenet = (len(hidden)-1)//2 
        self.latent = hidden[bottlenet]

        encoder_dims = [input_dim] + hidden[:bottlenet+1]
        decoder_dims = hidden[bottlenet:]
        self.encoder = _helper_net.MLP(encoder_dims, 
            use_batchnorm=True, use_dropout=0, output_activation="ReLU")
        

        self.decoder = nn.Sequential(
            _helper_net.MLP(decoder_dims, use_batchnorm=True, use_dropout=0),
            nn.Linear(hidden[-1], output_dim)
            )

        self.model = nn.Sequential(
             self.encoder,
             self.decoder
        )

    def encode(self, X):
        return self.encoder(X)

    def forward(self, X):
        z = self.encode(X)
        return self.decoder(z)

In [12]:
L_m = Linear_model(7341,4806, 3e-4,1e-3)

# training Linear model

In [13]:
Trainer = pl.Trainer(accelerator='gpu', gpus=[0],
                     default_root_dir = PATH.pth_dir,
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:467: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [ ]:
Trainer.fit(L_m, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

# training CAE model

In [40]:
VAE = CAE_model(7341,4806, hidden=[2048, 2048, 1536, 2048, 2048],
         lr=1e-4,weight_decay=1e-20)

In [ ]:
Trainer = pl.Trainer(accelerator='gpu', gpus=[0], auto_lr_find=True,
                     default_root_dir = PATH.pth_dir + '/quick_CVAE',
                     callbacks=[
                         callbacks.ModelCheckpoint(save_top_k=2, monitor="val_loss"),
                         callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
                         ],
                    )

Trainer.fit(VAE, train_dataloaders=train_dl, 
            val_dataloaders=test_dl)

In [14]:
L_M = Linear_model.load_from_checkpoint('/home/wergillius/data/diffuse_differentiate/lightning_logs/version_2/checkpoints/epoch=14-step=12165.ckpt',
                                        output_dim=4806, lr=3e-4, weight_decay=1e-3, input_dim=7341)

In [37]:
VAE = CAE_model.load_from_checkpoint('/home/wergillius/data/diffuse_differentiate/lightning_logs/version_7/checkpoints/epoch=24-step=20275.ckpt')

RuntimeError: Error(s) in loading state_dict for CAE_model:
	Missing key(s) in state_dict: "model.0.model.0.0.weight", "model.0.model.0.0.bias", "model.1.1.weight", "model.1.1.bias", "encoder.model.0.0.weight", "encoder.model.0.0.bias", "decoder.1.weight", "decoder.1.bias". 
	Unexpected key(s) in state_dict: "model.0.0.weight", "model.0.0.bias", "model.0.2.weight", "model.0.2.bias", "model.0.2.running_mean", "model.0.2.running_var", "model.0.2.num_batches_tracked", "model.1.0.weight", "model.1.0.bias", "encoder.0.weight", "encoder.0.bias", "encoder.2.weight", "encoder.2.bias", "encoder.2.running_mean", "encoder.2.running_var", "encoder.2.num_batches_tracked", "decoder.0.weight", "decoder.0.bias". 

In [12]:
VAE = CAE_model.load_from_checkpoint('/home/wergillius/data/diffuse_differentiate/quick_CVAE/lightning_logs/version_7/checkpoints/epoch=24-step=20275.ckpt')

In [21]:
Y_pred = []
with torch.no_grad():
    for X,Y in test_dl:
        Y_pred.append( L_M(X) )

In [13]:
VAE = VAE.eval().to('cpu');

Y_pred_CAE = []
with torch.no_grad():
    for X,Y in test_dl:
        Y_pred_CAE.append( VAE(X) )

In [14]:
ypred_CAE = torch.concat(Y_pred_CAE)
rdf_CAE = evaluate_r(ypred_CAE, Y_test)

In [17]:
rdf_CAE.to_csv("/home/wergillius/Project/diffuse_differentiate/result/TFAtlas/CAE_performance.csv")

In [22]:
ypred = torch.concat(Y_pred)
rdf = evaluate_r(ypred, Y_test)

In [24]:
rdf.to_csv("/home/wergillius/Project/diffuse_differentiate/result/TFAtlas/LinearModel_performance.csv")

In [66]:
ypred_L1 = torch.concat(Y_pred_L1)
rdf_L1 = evaluate_r(ypred_L1, Y_test)

In [67]:
rdf_L1

,gene,r2
0,A2M,0.081937
1,AADACL3,0.059327
2,ABCA1,0.389042
3,ABCA12,0.104837
4,ABCA13,0.064254
...,...,...
4801,ZRANB3,0.327969
4802,ZRSR2,0.210350
4803,ZSCAN1,0.157768
4804,ZSWIM6,0.281093


In [68]:
rdf.r2.max()

0.7024984154266845

In [44]:
ypred = torch.concat(Y_pred)

In [46]:
ypred.shape

torch.Size([2883, 4806])